# 📝 [제 5편] 선형 회귀 (1) - 오차와 손실함수

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「Linear Regression (I) - 오차 • 손실함수」 (ML_LEC_13), 「Linear Regression (II)」 (ML_LEC_14) 일부

---

## 🎯 학습 목표
1. 입력과 정답의 관계를 일차함수 y = Wx + b 로 나타내고, 가중치 W 와 바이어스 b 의 뜻을 말할 수 있다.
2. 오차 t − y 를 구하고, 오차를 제곱해서 평균 내는 이유를 설명할 수 있다.
3. 손실함수 E(W, b) 가 W, b 의 함수임을 알고, 손실함수가 가장 작아지는 직선을 찾는 것이 학습임을 이해한다.

---

## 🧭 지난 시간 복습
* 제 3편: 데이터 표를 입력 `x_data` 와 정답 `t_data` 로 나눴습니다.
* 제 4편: **회귀**는 공부 시간으로 시험 점수처럼 **연속적인 숫자**를 예측하는 지도학습입니다.

[⬅ 이전 : 제 4편 머신러닝의 개념과 종류](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/04_ml_overview.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 6편 선형 회귀 (2) - 경사하강법 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/06_gradient_descent.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

## 📖 개념 다지기 1. 데이터를 가장 잘 나타내는 직선 찾기

| 공부 시간 x | 9 | 14 | 21 | 27 | 32 | 37 |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 시험 성적 t | 74 | 81 | 86 | 90 | 88 | 92 |

* 공부 시간이 늘수록 성적도 대체로 오르는 경향 → **y = Wx + b** 꼴의 직선으로 나타낼 수 있습니다.
* 머신러닝에서는 기울기 W 를 **가중치(weight)**, y 절편 b 를 **바이어스(bias)** 라고 부릅니다.
* **학습(learning)** = 데이터를 가장 잘 나타내는 W 와 b 를 찾는 것

> **[생각해 보기]** 💡 아래 그래프의 세 직선 ①, ②, ③ 중 데이터를 가장 잘 나타내는 직선은? ( 　 )
> 그렇게 판단한 기준은 무엇인가요? (                                                            )

In [ ]:
x = np.array([9, 14, 21, 27, 32, 37], dtype=float)
t = np.array([74, 81, 86, 90, 88, 92], dtype=float)

xs = np.linspace(5, 40, 100)
plt.figure(figsize=(8, 5))
plt.scatter(x, t, s=80, color='royalblue', zorder=5, label='학습 데이터')
for (W, b, c, name) in [(1.0, 60, 'green', '①  y = 1.0x + 60'), (0.58, 71.6, 'crimson', '②  y = 0.58x + 71.6'), (0.3, 75, 'orange', '③  y = 0.3x + 75')]:
    plt.plot(xs, W*xs + b, color=c, lw=2, label=name)
plt.xlabel('공부 시간 (x)'); plt.ylabel('시험 성적 (t)'); plt.legend(); plt.grid(alpha=0.3)
plt.title('어느 직선이 가장 잘 맞을까?'); plt.show()

## 📖 개념 다지기 2. 오차(error) = 정답 − 예측값

직선 y = Wx + b 로 계산한 값을 **예측값 y**, 실제 값을 **정답 t** 라고 하면

> **오차 = t − y = t − (Wx + b)**

**[연습 1]** 직선 y = 0.5x + 70 일 때 표를 완성하세요.

| x | 9 | 14 | 21 | 27 | 32 | 37 |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 정답 t | 74 | 81 | 86 | 90 | 88 | 92 |
| 예측 y = 0.5x + 70 | 74.5 | 77 | ( 　 ) | 83.5 | ( 　 ) | 88.5 |
| 오차 t − y | −0.5 | 4 | ( 　 ) | 6.5 | ( 　 ) | 3.5 |
| 오차² | 0.25 | 16 | ( 　 ) | 42.25 | ( 　 ) | 12.25 |

In [ ]:
W, b = 0.5, 70
y = W*x + b
print('예측 y   =', y)
print('오차 t-y =', t - y)
print('오차²    =', (t - y)**2)

## 📖 개념 다지기 3. 왜 오차를 '제곱' 할까?

오차를 그냥 더하면 **+ 오차와 − 오차가 서로 지워져서** 합이 0 에 가까워질 수 있습니다.
→ 합이 0 이라고 해서 정말 잘 맞는 직선인지 알 수 없습니다.

그래서 **오차를 제곱**해서 더합니다.
1. 제곱하면 언제나 **0 이상** → 서로 지워지지 않는다.
2. 오차가 큰 점은 제곱하면 **훨씬 더 커진다** (2² = 4, 10² = 100) → 많이 틀린 점을 더 강하게 고치게 된다.

> **[확인 1]** 오차가 (+3, −3) 인 두 점이 있다. 오차의 합 = ( 　 ),  오차²의 합 = ( 　 )

## 📖 개념 다지기 4. 손실함수 (loss function)

데이터가 n 개일 때, 오차² 의 **평균**을 손실함수라고 합니다.

> **E(W, b) = {(t₁ − y₁)² + (t₂ − y₂)² + … + (tₙ − yₙ)²} / n**
> 　　　　 = (1/n) × Σ<sub>i=1</sub><sup>n</sup> {t<sub>i</sub> − (W x<sub>i</sub> + b)}²

* x 와 t 는 데이터로 **이미 주어진 값**입니다.
* 따라서 E 의 값을 바꾸는 것은 **W 와 b 뿐** → 손실함수는 **W, b 의 함수** E(W, b)

> **[연습 2]** [연습 1] 의 직선 y = 0.5x + 70 의 손실함수 값은
> E(0.5, 70) = (0.25 + 16 + 30.25 + 42.25 + 4 + 12.25) / 6 = ( 　　 )

> 🎯 **선형 회귀의 최종 목표**: 손실함수 E(W, b) 가 **가장 작아지는** W 와 b 를 찾는 것!

In [ ]:
def loss_func(W, b, x, t):
    y = W*x + b
    return np.mean((t - y)**2)          # 오차² 의 평균

print('E(0.5, 70)  =', loss_func(0.5, 70, x, t))
print('E(1.0, 60)  =', loss_func(1.0, 60, x, t))
print('E(0.58, 71.6) =', round(loss_func(0.58, 71.6, x, t), 4))

> **[확인 2]** 세 직선 중 손실함수 값이 가장 작은 직선은 ( 　 ) 번이고, 이것이 처음 그래프에서 고른 직선과 같은가요?

## 🎬 애니메이션 1. '오차²' 은 진짜 정사각형의 넓이다!

오차가 t − y 이면 오차² 은 **한 변이 |t − y| 인 정사각형의 넓이** 입니다.
기울기 W 를 0 → 1.2 로 바꿀 때 정사각형들의 넓이(빨강)와 손실함수 E(W) (오른쪽)가 어떻게 변하는지 보세요.
(이 애니메이션에서는 바이어스를 b = 71.6 으로 고정합니다)

In [ ]:
#@title 🎬 애니메이션 1 실행 (오차 정사각형과 손실함수)
b_fix = 71.6
Ws = np.concatenate([np.linspace(0.0, 1.2, 50), np.linspace(1.2, 0.58, 25), np.full(10, 0.58)])
W_curve = np.linspace(0.0, 1.2, 200)
E_curve = [loss_func(w, b_fix, x, t) for w in W_curve]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.5))
def draw(i):
    W = Ws[i]; y = W*x + b_fix; E = loss_func(W, b_fix, x, t)
    axL.clear(); axR.clear()
    xs = np.linspace(0, 45, 50)
    axL.plot(xs, W*xs + b_fix, color='crimson', lw=2, label=f'y = {W:.2f}x + {b_fix}')
    for xi, ti, yi in zip(x, t, y):
        d = ti - yi
        axL.plot([xi, xi], [yi, ti], 'k:', lw=1)
        # 오차를 한 변으로 하는 정사각형 (x 축, y 축 눈금이 같도록 aspect='equal')
        axL.add_patch(plt.Rectangle((xi, min(yi, ti)), abs(d), abs(d), fc='red', alpha=0.25, ec='red'))
    axL.scatter(x, t, s=60, color='royalblue', zorder=5, label='정답 t')
    axL.set_xlim(0, 45); axL.set_ylim(60, 115); axL.set_aspect('equal')
    axL.set_xlabel('공부 시간 x'); axL.set_ylabel('성적'); axL.legend(loc='upper left')
    axL.set_title(f'빨간 정사각형 넓이의 평균 = E = {E:.2f}', fontsize=13)
    axR.plot(W_curve, E_curve, color='purple', lw=2)
    axR.plot([W], [E], 'o', color='crimson', ms=11)
    axR.set_xlabel('가중치 W'); axR.set_ylabel('손실함수 E(W)'); axR.grid(alpha=0.3)
    axR.set_title('W 에 따른 손실함수 E(W)  (b = 71.6 고정)', fontsize=13)
    axR.set_ylim(0, max(E_curve)*1.05)
anim = animation.FuncAnimation(fig, draw, frames=len(Ws), interval=100)
plt.close(fig)
show_anim(anim)

> **[관찰 1]**
> ① 손실함수 E(W) 의 그래프는 ( 　　　　 ) 모양이다. (힌트: 수학 시간에 배운 함수)
> ② 정사각형들이 가장 작아질 때의 W 는 약 ( 　　 ) 이고, 이때 그래프는 ( 꼭짓점 / 양 끝 ) 에 있다.

## 🎛️ 직접 조절해 보기

슬라이더로 W 와 b 를 바꿔 가며 **손실함수 값이 가장 작아지는 직선**을 찾아보세요. 친구와 누가 더 작은 E 를 찾는지 겨뤄 보세요!

> 내가 찾은 값:  W = ( 　　 ),  b = ( 　　 ),  E = ( 　　 )

In [ ]:
#@title 🎛️ 슬라이더 실행
import ipywidgets as widgets
def show(W=0.5, b=70.0):
    y = W*x + b; E = loss_func(W, b, x, t)
    fig, ax = plt.subplots(figsize=(7, 5))
    xs = np.linspace(0, 45, 50)
    ax.plot(xs, W*xs + b, color='crimson', lw=2)
    for xi, ti, yi in zip(x, t, y):
        ax.add_patch(plt.Rectangle((xi, min(yi, ti)), abs(ti - yi), abs(ti - yi), fc='red', alpha=0.25, ec='red'))
    ax.scatter(x, t, s=60, color='royalblue', zorder=5)
    ax.set_xlim(0, 45); ax.set_ylim(60, 115); ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_title(f'y = {W:.2f}x + {b:.1f}    →    E(W, b) = {E:.3f}', fontsize=14)
    plt.show()
widgets.interact(show, W=widgets.FloatSlider(value=0.5, min=0, max=1.5, step=0.01),
                       b=widgets.FloatSlider(value=70, min=50, max=90, step=0.1))

## 📖 개념 다지기 5. 손실함수의 모양 - 간단한 데이터로 손 계산

계산을 쉽게 하기 위해 아래 데이터에서 **b = 0** 으로 두고 E(W) 를 계산해 봅시다.

| x | 1 | 2 | 3 |
|:-:|:-:|:-:|:-:|
| t | 1 | 2 | 3 |

E(W) = {(1 − W·1)² + (2 − W·2)² + (3 − W·3)²} / 3 = {(1 − W)² × (1 + 4 + 9)} / 3 = (14/3)(1 − W)²

| W | −1 | 0 | 1 | 2 | 3 |
|:-:|:-:|:-:|:-:|:-:|:-:|
| E(W) | 18.7 | ( 　 ) | ( 　 ) | 4.67 | ( 　 ) |

> **[생각해 보기]** 💡 E(W) = (14/3)(1 − W)² 은 W = ( 　 ) 에서 최솟값 ( 　 ) 을 갖는 이차함수이다.
> 이번처럼 식이 간단하면 손으로 꼭짓점을 찾을 수 있지만, 변수(W)가 수백만 개인 인공지능은 어떻게 최솟값을 찾을까요?
> → **다음 시간: 미분을 이용한 경사하강법!**

In [ ]:
x2 = np.array([1., 2., 3.]); t2 = np.array([1., 2., 3.])
for W in [-1, 0, 1, 2, 3]:
    print(f'W = {W:2d}  →  E(W) = {loss_func(W, 0, x2, t2):.2f}')

Wg = np.linspace(-1.2, 3.2, 200)
plt.figure(figsize=(6, 4))
plt.plot(Wg, [loss_func(w, 0, x2, t2) for w in Wg], color='purple', lw=2)
plt.scatter([-1, 0, 1, 2, 3], [loss_func(w, 0, x2, t2) for w in [-1, 0, 1, 2, 3]], color='crimson', zorder=5)
plt.xlabel('W'); plt.ylabel('E(W)'); plt.title('E(W) = (14/3)(1 − W)²'); plt.grid(alpha=0.3); plt.show()

## 🎬 애니메이션 2. W 와 b 를 함께 바꾸면? - 손실함수 '그릇'

W 와 b 가 둘 다 변하면 E(W, b) 는 **길쭉한 그릇(골짜기) 모양의 곡면**이 됩니다. 그릇의 **가장 낮은 바닥(빨간 점)** 이 우리가 찾는 W, b 입니다.
(공부 시간 데이터 사용. 왼쪽 곡면이 천천히 돌아가고, 오른쪽은 위에서 내려다본 등고선입니다. 너무 높은 부분은 잘라서 보여 줍니다)

In [ ]:
#@title 🎬 애니메이션 2 실행 (3D 손실함수)
Wg, bg = np.meshgrid(np.linspace(0.2, 1.0, 70), np.linspace(60, 84, 70))
Eg = np.mean((t[None, None, :] - (Wg[..., None]*x[None, None, :] + bg[..., None]))**2, axis=-1)
Ec = np.minimum(Eg, 120)                                  # 높은 부분 자르기
best = np.unravel_index(np.argmin(Eg), Eg.shape)

fig = plt.figure(figsize=(13, 5.5))
ax = fig.add_subplot(121, projection='3d'); ax2 = fig.add_subplot(122)
cs = ax2.contourf(Wg, bg, Ec, levels=20, cmap='viridis')
ax2.plot(Wg[best], bg[best], 'r*', ms=18)
ax2.set_xlabel('W'); ax2.set_ylabel('b'); ax2.set_title('위에서 본 모습 (등고선)', fontsize=13)
def draw(i):
    ax.clear()
    ax.plot_surface(Wg, bg, Ec, cmap='viridis', alpha=0.9, linewidth=0)
    ax.scatter([Wg[best]], [bg[best]], [Eg[best]], color='red', s=80)
    ax.set_xlabel('W'); ax.set_ylabel('b'); ax.set_zlabel('E(W, b)'); ax.set_zlim(0, 120)
    ax.set_title(f'가장 낮은 곳 : W 약 {Wg[best]:.2f}, b 약 {bg[best]:.1f}', fontsize=13)
    ax.view_init(elev=35, azim=-60 + i*9)
anim = animation.FuncAnimation(fig, draw, frames=40, interval=150)
plt.close(fig)
show_anim(anim)

## ✏️ 연습 문제

> **[문제 1]** 직선 y = x + 60 의 공부 시간 데이터에 대한 오차를 모두 구하고, 오차의 합과 손실함수 값을 구하세요.
> 오차: ( 　, 　, 　, 　, 　, 　 ),  오차의 합 = ( 　 ),  E = ( 　　 )

> **[문제 2]** [문제 1] 의 직선과 y = 0.5x + 70 중 어느 쪽이 더 좋은 직선인가요? **오차의 합**으로 판단할 때와 **손실함수**로 판단할 때 결과가 같은지 비교하세요.

아래 셀의 `______` 를 채워 확인하세요.

In [ ]:
W, b = ______, ______
err = t - (W*x + b)
print('오차      =', err)
print('오차의 합 =', ______)
print('손실함수  =', loss_func(W, b, x, t))

In [ ]:
#@title 🔑 정답 확인 (먼저 풀어 본 뒤에 실행하세요) { display-mode: "form" }
for (W, b) in [(1.0, 60), (0.5, 70)]:
    err = t - (W*x + b)
    print(f'y = {W}x + {b} :  오차 = {err.tolist()},  오차의 합 = {err.sum():.1f},  E = {loss_func(W, b, x, t):.2f}')
print('→ 오차의 합으로는 y = x + 60 (11) 이 더 좋아 보이지만, 손실함수로는 y = 0.5x + 70 (17.5) 이 더 좋다.')
print('  y = x + 60 은 +오차와 −오차가 서로 지워져 합이 작아 보였을 뿐이다!')

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 모델 | y = Wx + b  (W: 가중치, b: 바이어스) |
| 오차 | t − y = t − (Wx + b) |
| 손실함수 | E(W, b) = (1/n) × Σ {t<sub>i</sub> − (W x<sub>i</sub> + b)}²  ← 오차² 의 평균 |
| 학습 목표 | E(W, b) 가 최소가 되는 W, b 찾기 |

### 🏆 도전 과제
E(W) 그래프에서 현재 W = 3 에 있다고 하자. 이 점에서 접선의 기울기는 ( 양수 / 음수 ) 이므로,
E(W) 를 줄이려면 W 를 ( 늘려야 / 줄여야 ) 한다. → 제 2편의 수치미분으로 이 기울기를 구할 수 있다!

---

[⬅ 이전 : 제 4편 머신러닝의 개념과 종류](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/04_ml_overview.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 6편 선형 회귀 (2) - 경사하강법 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/06_gradient_descent.ipynb)